In [31]:
# Import Libraries

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

In [32]:
# Upload Dataset

from google.colab import files
uploaded = files.upload()

Saving processed-data.csv to processed-data (1).csv


In [33]:
# Load Dataset

df = pd.read_csv("processed-data.csv")
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,3,4,1,6,3,3,2,2,2,2


In [34]:
# Convert Target Variable

df["Attrition"] = df["Attrition"].map({"Yes": 1, "No": 0})
print("Attrition distribution:\n", df["Attrition"].value_counts())

Attrition distribution:
 Attrition
0    1233
1     237
Name: count, dtype: int64


In [44]:
# One-Hot Encoding FIRST

categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
print("\nEncoding columns:", categorical_cols)
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print("Shape after encoding:", df.shape)


Encoding columns: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
Shape after encoding: (1470, 51)


In [45]:
# ── Feature Engineering (must match backend loader.py exactly) ──

# 1. Income Per Year

df["IncomePerYear"] = df["MonthlyIncome"] * 12

In [46]:
# 2. Promotion Gap

df["PromotionGap"] = df["YearsSinceLastPromotion"] / df["YearsAtCompany"].replace(0, 1)

In [47]:
# 3. Company Experience Ratio

df["CompanyExperienceRatio"] = df["YearsAtCompany"] / df["TotalWorkingYears"].replace(0, 1)

In [48]:
# 4. Manager Stability

df["ManagerStability"] = df["YearsWithCurrManager"] / df["YearsAtCompany"].replace(0, 1)

In [49]:
# 5. Workload Score — uses encoded columns

ot_col = "OverTime_Yes" if "OverTime_Yes" in df.columns else None
bt_freq = "BusinessTravel_Travel_Frequently" if "BusinessTravel_Travel_Frequently" in df.columns else None
bt_rare = "BusinessTravel_Travel_Rarely" if "BusinessTravel_Travel_Rarely" in df.columns else None

print("\nOverTime column found:", ot_col)
print("BusinessTravel_Frequently found:", bt_freq)
print("BusinessTravel_Rarely found:", bt_rare)

df["WorkloadScore"] = (
    (df[ot_col].astype(int) * 2 if ot_col else 0) +
    (df[bt_freq].astype(int) * 2 if bt_freq else 0) +
    (df[bt_rare].astype(int) if bt_rare else 0)
)


OverTime column found: OverTime_Yes
BusinessTravel_Frequently found: BusinessTravel_Travel_Frequently
BusinessTravel_Rarely found: BusinessTravel_Travel_Rarely


In [50]:
# 6. Salary Growth

df["SalaryGrowth"] = df["PercentSalaryHike"] / df["YearsAtCompany"].replace(0, 1)
print("\nAll engineered features added successfully")
print("Shape after feature engineering:", df.shape)


All engineered features added successfully
Shape after feature engineering: (1470, 51)


In [51]:
# Separate Features and Target

X = df.drop("Attrition", axis=1)
y = df["Attrition"]

print(f"\nFeature count: {X.shape[1]}")
print("\nAll feature columns:")
for i, col in enumerate(X.columns.tolist()):
    print(f"  {i+1}. {col}")


Feature count: 50

All feature columns:
  1. Age
  2. DailyRate
  3. DistanceFromHome
  4. Education
  5. EnvironmentSatisfaction
  6. HourlyRate
  7. JobInvolvement
  8. JobLevel
  9. JobSatisfaction
  10. MonthlyIncome
  11. MonthlyRate
  12. NumCompaniesWorked
  13. PercentSalaryHike
  14. PerformanceRating
  15. RelationshipSatisfaction
  16. StockOptionLevel
  17. TotalWorkingYears
  18. TrainingTimesLastYear
  19. WorkLifeBalance
  20. YearsAtCompany
  21. YearsInCurrentRole
  22. YearsSinceLastPromotion
  23. YearsWithCurrManager
  24. IncomePerYear
  25. PromotionGap
  26. CompanyExperienceRatio
  27. ManagerStability
  28. WorkloadScore
  29. SalaryGrowth
  30. BusinessTravel_Travel_Frequently
  31. BusinessTravel_Travel_Rarely
  32. Department_Research & Development
  33. Department_Sales
  34. EducationField_Life Sciences
  35. EducationField_Marketing
  36. EducationField_Medical
  37. EducationField_Other
  38. EducationField_Technical Degree
  39. Gender_Male
  40. JobRo

In [52]:
# Save feature_columns.pkl

feature_columns = X.columns.tolist()
joblib.dump(feature_columns, "feature_columns.pkl")
print(f"\nfeature_columns.pkl saved — {len(feature_columns)} features")


feature_columns.pkl saved — 50 features


In [53]:
# Fit and Save Scaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, "scaler.pkl")
print("scaler.pkl saved")

scaler.pkl saved


In [54]:
# Save Scaled Dataset

df_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=df.index)
df_scaled["Attrition"] = y.values
df_scaled.to_csv("feature-engineered-data.csv", index=False)
print("feature-engineered-data.csv saved")

feature-engineered-data.csv saved


In [55]:
# Final Verification

print(f"\n{'='*40}")
print(f"Final dataset shape: {df_scaled.shape}")
print(f"Feature count: {X.shape[1]}")
print(f"Target distribution: {df_scaled['Attrition'].value_counts().to_dict()}")
print(f"{'='*40}")


Final dataset shape: (1470, 51)
Feature count: 50
Target distribution: {0: 1233, 1: 237}


In [56]:
# Download all artifacts

from google.colab import files
files.download("feature_columns.pkl")
files.download("scaler.pkl")
files.download("feature-engineered-data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>